In [ ]:
# Import libs
import numpy as np
import pandas as pd
from pandas.plotting import parallel_coordinates

import os
import sqlite3
import math
from collections import Counter
from pathlib import Path
import imblearn

# Encoding data
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# Visualization
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Model
from scipy.stats import skew
import yellowbrick
import sklearn
from sklearn.decomposition import PCA 
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE 
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

# Metrics
from sklearn.metrics import RocCurveDisplay # For ROC curve
from sklearn.metrics import PrecisionRecallDisplay # For pr 
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_recall_fscore_support


# Config
mpl.rcParams['font.family'] = 'monospace' 
sns.set_theme(style="white", palette=None)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

sns.set(font = 'Serif', style = 'white', rc = {'axes.facecolor':'#fafafa', 'figure.facecolor':'#fafafa'})

# 1. Load data

**Read dataset via pandas.read_csv()**

In [ ]:
df = pd.read_csv(r'../input/breast-cancer-dataset/BRCA.csv')
df.head(10)

**Check for missing values**

In [ ]:
na = df.isna().sum()
na[na>0]

We should drop all the rows where Patient_Status(the target variable) is null

In [ ]:
df = df[~df["Patient_Status"].isna()]

# 2. Quick EDA 

**Descriptive numeric variables**

In [ ]:
df.describe().round(2).T

- The mean age is 58, indicating that there are quite a few elderly patients in this breast cancer data set

In [ ]:
num_cols = df.describe().columns.values
cols = df.columns.values
cat_cols = ['Tumour_Stage','Histology','ER status','PR status','HER2 status','Surgery_type']

**Look at gender**

We should study the cancer index of female patients only, right?

In [ ]:
df.Gender.value_counts()

- There seems to be a big difference in gender, the dataset seems biased so we don't consider gender as an input variable for the model.
- I decided to drop all male patients from this dataset.

In [ ]:
df = df[df["Gender"]=="FEMALE"]

**Now, look at the target: Patient Status - Dead or Alive!**

In [ ]:
target = 'Patient_Status'
df[target].value_counts().plot(kind='bar')

The target! About 20% patients dead

**Let's explore the distribution of numerical variables!**

In [ ]:
fig, ax = plt.subplots(nrows = 2, ncols = 3, figsize = (15, 8), constrained_layout = True) 

fig.suptitle("Histogram plot on Numerical values", fontsize=16, fontweight='bold') # title

gs = fig.add_gridspec(3,3)
gs.update(wspace=0.8, hspace=0.5)
background_color ='#fafafa'

ax0 = ax[0,0]
ax0.set_axis_off()
ax = np.delete(ax.ravel(),0)

for axes,num in zip(ax.ravel(),num_cols):
    sns.histplot(x = num, data = df, ax = axes, kde = True, alpha =0.9, color = 'navy')
    axes.set_ylabel('')
    axes.grid(which='major',ls='-.',axis='y')
    
for axes in ax.ravel():
    for i in ['top','left','right']:
        axes.spines[i].set_visible(False)
    

As we can see from these graphs, Protein1 and Protein4 mabe skewed to the left while Age and Protein3 is skewed to the right.

In [ ]:
import textwrap

fig, ax = plt.subplots(nrows = 2, ncols = 3, figsize = (14, 8), constrained_layout = True) # axis.patches can't be used
gs = fig.add_gridspec(3,3)
gs.update(wspace=1, hspace=1)
background_color ='#fafafa'
color_palette = ['green','red']

for axes,cat in zip(ax.ravel(),cat_cols):
    #df_ = df.groupby(cat)[cat]
    sns.countplot(data = df,
                x = cat,
                ax = axes,
                hue='Patient_Status',
                palette = color_palette)
    
    axes.set_ylabel('Total Count')
    axes.grid(which='major',ls='-.',axis='y')
    
for axes in ax.ravel():
    for i in ['top','left','right']:
        axes.spines[i].set_visible(False)
    # Wrap xticklabels :
    labels = [textwrap.fill(label.get_text(), 15) for label in axes.get_xticklabels()]
    axes.set_xticklabels(labels)

In [ ]:
df_corr = df[num_cols].corr().transpose()

fig = plt.figure(figsize=(10,6))
gs = fig.add_gridspec(1,1)
gs.update(wspace=0.3, hspace=0.15)
ax0 = fig.add_subplot(gs[0,0])

color_palette = ["#5833ff","#da8829"]
mask = np.triu(np.ones_like(df_corr))
ax0.text(1.5,-0.1,"Correlation Matrix",fontsize=14, fontweight='bold', fontfamily='serif', color="#000000")
sns.heatmap(df_corr,mask=mask,fmt=".3f",annot=True,cmap='Blues')
plt.show()

# 3. Build model

**Dummies variable**

In [ ]:
df_train = df.copy()
df_train = pd.get_dummies(df_train, columns = cat_cols, drop_first = True)
df_train.head()

**Encoding labels**

We have to encode the target Patient Status that "0" indicates "Alive" and "1" indicates "Dead"

In [ ]:
# Encoding labels 
df_train['Patient_Status'] = df_train['Patient_Status'].apply(lambda x: 0 if x=="Alive" else 1)
df_train['Patient_Status'].value_counts()

**Drop unnecessary columns**

In [ ]:
X = df_train.set_index("Patient_ID").drop(['Gender','Date_of_Surgery','Date_of_Last_Visit','Patient_Status'],axis=1).copy()
y = df_train['Patient_Status'].copy()

**Split train, test**
> Test size = 0.2*sample size , I also set stratify=y to balance the number of Alive and Dead status in train and test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 12, stratify=y)
print("The shape of X_train is ", X_train.shape)
print("The shape of X_test is ",X_test.shape)
print("The shape of y_train is ",y_train.shape)
print("The shape of y_test is ",y_test.shape)

> Use SMOTE algorithm to over-sampling the minorities

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy = 'not majority')
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(Counter(y_train))
print(Counter(y_train_smote))

**Try Support Vector Machine with over-sampling train set**

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

svc = SVC(kernel='rbf', C=.01, gamma='auto')

svc.fit(X_train_smote, y_train_smote)
y_pred = svc.predict(X_test)

print("The test accuracy score of SVM is ", accuracy_score(y_test, y_pred))
print("========================\n", classification_report(y_test, y_pred))
print("Cross_val_score:", cross_val_score(svc, X_train, y_train, scoring='f1_macro'))

**Try some imblearn classifiers**

In [ ]:
from imblearn.ensemble import RUSBoostClassifier, EasyEnsembleClassifier,BalancedBaggingClassifier

def classify_report_(model, X_train, y_train):
    clf = model
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    return classification_report(y_test, y_pred), np.mean(cross_val_score(model, X_train, y_train))

In [ ]:
models = [RUSBoostClassifier(random_state=30),
          EasyEnsembleClassifier(random_state=30),
          BalancedBaggingClassifier(random_state=30)]
for m in models:
    print('=====================================\n',str(m))
    print(classify_report_(m, X_train, y_train)[0])
    print("Mean F1-macro:", classify_report_(m, X_train, y_train)[1])

Poor f1-score caused by imbalance dataset

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier(class_weight="balanced", random_state=32)
print(classify_report_(dtc, X_train, y_train)[0])

In [ ]:
features_importance = dict(zip(dtc.feature_names_in_, np.round(dtc.feature_importances_,2)))
print(features_importance)

Select importance features and fit back to the model

In [ ]:
selected_features = {k: v for k, v in features_importance.items() if v >= 0.1}
selected_features = list(selected_features.keys())
print(selected_features)

In [ ]:
X_train, X_test = X_train[selected_features], X_test[selected_features]
X_train_smote = X_train_smote[selected_features]

In [ ]:
from sklearn.model_selection import StratifiedKFold
# Try tuning DecisionTreeClassifier model 
param_grid = {'criterion':['gini','entropy'],
              'max_depth': np.arange(2,10, step=2),
              'min_samples_split': np.arange(2,10, step=2),
              'max_features': ['auto', 'sqrt', 'log2', None],
              'class_weight': ['balanced', {1:4}, {1:3}]
             }
print(param_grid)

In [ ]:
cv = StratifiedKFold(n_splits=3)

dtc_searcher = GridSearchCV(
    estimator = dtc,
    param_grid = param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=2,
    verbose=1
)

dtc_searcher.fit(X_train, y_train) 

In [ ]:
print(classification_report(y_test, dtc_searcher.predict(X_test)))

Mayn't be good to keep the best estimators returned by grid_search  
Overall, I'll choose **DecisionTreeClassifier(class_weight="balanced")** to predict

In [ ]:
df_train = df_train.set_index("Patient_ID")

In [ ]:
df_test = df_train[df_train.index.isin(X_test.index)]

In [ ]:
dtc = DecisionTreeClassifier(class_weight="balanced", random_state=32)
dtc.fit(X_train, y_train)
y_pred = dtc.predict(X_test)

In [ ]:
df_test["Predict"] = dtc.predict(X_test).tolist()
df_test["Actual"] = y_test.tolist()
df_test.to_csv("./Dead_Probability.csv")